# Adversarial Attacks on Image Classifiers: From FGSM to Passive Defense

A **small, human-imperceptible perturbation** added to a correctly classified image can make a fixed classifier output a completely wrong label. Here a pretrained **ResNet-50** confidently calls a photo a *Tiger Cat* (p ≈ 0.64) — then, after we add a perturbation invisible to the human eye, the **same network** outputs *Star Fish* or *Keyboard* with confidence ≈ 1.0.

Everything runs on a **pretrained ImageNet classifier with no training** — each claim is something you run and see.

## Roadmap — six movements

1. **Define** — what is an adversarial example? targeted vs. non-targeted *(LO1)*
2. **Formalize** — the attack as constrained optimization of an input-space loss *(LO2)*
3. **Measure** — L2 vs. L∞ and why L∞ captures imperceptibility *(LO3)*
4. **Attack** — the Fast Gradient Sign Method, FGSM *(LO4)*
5. **Strengthen** — iterative FGSM with projection back into the ε-ball *(LO5)*
6. **Defend** — a passive input-transformation defense and its accuracy trade-off *(LO6)*

> **Demo philosophy:** one frozen classifier; gradients taken w.r.t. the *input*, not the weights. No model is ever trained — we only craft inputs and filter them.

## Part 0 — Setup

This demo uses a **pretrained ImageNet classifier** and PyTorch **autograd on the input tensor** — the lecture's framing of taking the gradient of the loss with respect to the *input*, not the parameters. It runs on a free Colab **CPU or GPU** runtime and **requires no training**.

The pretrained ResNet-50 weights and a sample image are **downloaded at runtime and cached**, so the first run of the setup cells may take a few seconds.

In [ ]:
# Colab fallback (uncomment if ipywidgets is missing):
# !pip install -q ipywidgets

import torch
import torch.nn.functional as F
import torchvision
from torchvision import models, transforms
import torchvision.transforms.functional as TF
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
import urllib.request

# ipywidgets is optional: guard the import so the notebook still runs statically.
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except Exception:
    widgets = None
    HAS_WIDGETS = False
    print('[warn] ipywidgets unavailable - interactive cells will render a static default.')
    print('       Enable it in Colab with:  !pip install -q ipywidgets')

# scipy is an optional fallback for Gaussian smoothing in the defense cells.
try:
    import scipy.ndimage as ndi
    HAS_SCIPY = True
except Exception:
    ndi = None
    HAS_SCIPY = False

# Enable inline plotting when running inside an IPython kernel.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# Device: GPU if available, else CPU (CPU is fine for single-image attacks).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Fix all RNG seeds so attacks and visualizations are reproducible across Run All.
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Consistent matplotlib defaults for every image panel and bar chart below.
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.titlesize'] = 13
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['axes.grid'] = False

assert device.type in ('cuda', 'cpu')
print(f'torch {torch.__version__} | device: {device} | widgets: {HAS_WIDGETS} | scipy: {HAS_SCIPY}')

In [ ]:
# ---- Shared toolkit: frozen classifier, labels, normalization, helpers (defined once) ----

# 1) Load a pretrained ResNet-50 and freeze it: f(x) is fixed for the whole notebook.
try:
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)
except Exception as e:
    raise RuntimeError('Could not download pretrained ResNet-50 weights. Re-run with internet enabled in Colab.') from e

model.eval().to(device)
for p in model.parameters():
    p.requires_grad_(False)  # attacks differentiate w.r.t. the INPUT only, never the weights

# 2) ImageNet class labels (1000 strings), with a download fallback for older torchvision.
try:
    imagenet_labels = list(weights.meta['categories'])
except Exception:
    import json
    _url = 'https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json'
    with urllib.request.urlopen(_url, timeout=10) as r:
        _idx = json.load(r)
    imagenet_labels = [_idx[str(i)][1] for i in range(1000)]
assert len(imagenet_labels) == 1000, f'expected 1000 labels, got {len(imagenet_labels)}'

# 3) ImageNet normalization as (1,3,1,1) tensors + normalize/denormalize closures.
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def normalize(pix):
    '''Pixel-space [0,1] tensor -> normalized tensor the model expects.'''
    return (pix - mean) / std

def denormalize(norm):
    '''Normalized tensor -> pixel-space [0,1] tensor.'''
    return norm * std + mean

# 4) Preprocess a PIL image into a normalized (1,3,224,224) tensor on device.
_preprocess_pil = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

def preprocess(pil_img):
    pix = _preprocess_pil(pil_img.convert('RGB')).unsqueeze(0).to(device)
    return normalize(pix)

# 5) Convert a normalized tensor back to a displayable HWC image in [0,1].
def to_displayable(x):
    img = denormalize(x.detach().to('cpu'))
    img = img.clamp(0, 1).squeeze(0).permute(1, 2, 0).numpy()
    return img

# 6) Top-k prediction: returns (labels, probabilities, indices).
def predict(x, k=5):
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        top_p, top_i = probs.topk(k, dim=1)
    idxs = top_i[0].tolist()
    return [imagenet_labels[i] for i in idxs], top_p[0].tolist(), idxs

# 7) Horizontal probability bar chart (reused by the baseline and every attack result).
def plot_topk(labels, probs, title='', ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 3))
    y = np.arange(len(labels))
    ax.barh(y, probs, color='#4C72B0')
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel('probability')
    ax.set_title(title)
    for yi, prob in zip(y, probs):
        ax.text(min(prob + 0.02, 0.98), yi, f'{prob:.2f}', va='center', fontsize=9)
    return ax

# 8) Sanity check: 1000-class logits and a populated label list.
with torch.no_grad():
    _out = model(torch.zeros(1, 3, 224, 224, device=device))
assert _out.shape == (1, 1000)
print(f'ResNet-50 ready on {device} | {len(imagenet_labels)} ImageNet labels | logits {tuple(_out.shape)}')

In [ ]:
# ---- Acquire the running-example image (the canonical Tiger Cat) robustly ----
IMAGE_URL = 'https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n02123159_tiger_cat.JPEG'
FALLBACK_URL = 'https://github.com/EliSchwartz/imagenet-sample-images/raw/master/n02123159_tiger_cat.JPEG'

def load_sample_image(url, timeout=10.0):
    '''Download an image and return it as an RGB PIL image.'''
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        data = resp.read()
    return Image.open(io.BytesIO(data)).convert('RGB')

orig_image = None
source_used = None
for _name, _url in [('primary', IMAGE_URL), ('mirror', FALLBACK_URL)]:
    try:
        orig_image = load_sample_image(_url)
        source_used = _name
        break
    except Exception as e:
        print(f'[warn] {_name} image download failed: {e}')

if orig_image is None:
    # Final offline fallback: a deterministic synthetic image so the notebook never hard-fails.
    rng = np.random.default_rng(SEED)
    arr = rng.integers(0, 256, size=(224, 224, 3)).astype('uint8')
    yy, xx = np.mgrid[0:224, 0:224]
    arr[..., 0] = ((np.sin(xx / 18.0) + 1) * 110).astype('uint8')
    arr[..., 1] = ((np.cos(yy / 22.0) + 1) * 110).astype('uint8')
    orig_image = Image.fromarray(arr, 'RGB')
    source_used = 'synthetic'
    print('[warn] Using a SYNTHETIC fallback image: predictions will differ from the canonical Tiger Cat.')

x0 = preprocess(orig_image)
print(f'image source: {source_used} | size: {orig_image.size} | x0: {tuple(x0.shape)} on {x0.device}')

## 1 · What is an adversarial example?

Take a benign image **x**, add a tiny perturbation **Δx**, and feed **x + Δx** to a **fixed** classifier f (the weights never change):

- **Non-targeted attack** — make the model output **anything but** the true label.
- **Targeted attack** — force the model to output a **specific chosen** wrong label.

The perturbation is constrained to be small enough that **a human sees no difference**. In the lecture's example the benign *Tiger Cat* (p ≈ 0.64) becomes *Star Fish* / *Keyboard* (p ≈ 1.0); the perturbation only looks like noise when **amplified ~50×**.

In [ ]:
# ---- Benign baseline: what the fixed classifier predicts on the clean image ----
labels, probs, idxs = predict(x0, k=5)
y_hat = idxs[0]  # the model OWN top-1 class index (self-consistent reference, not ground truth)

fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 4))
ax_img.imshow(to_displayable(x0))
ax_img.set_title(f'Benign: {labels[0]} ({probs[0]:.2f})')
ax_img.axis('off')
plot_topk(labels, probs, 'Top-5 predictions', ax=ax_bar)
plt.tight_layout()
benign_baseline_figure = fig
plt.show()

if probs[0] < 0.30:
    print('[note] Top-1 confidence is low - the sample may be the synthetic fallback, so later narration is illustrative.')
print(f'y_hat = {y_hat}  ({imagenet_labels[y_hat]})  |  confidence {probs[0]:.3f}')

## 2 · The attack objective (constrained optimization)

Every concrete attack solves the same problem — fool the classifier **while staying close to the original**:

$$x^* = \arg\min_{d(x_0, x) \le \varepsilon} L(x)$$

- **Non-targeted:** $L(x) = -e(y, \hat{y})$ — push the output **away** from the true label $\hat{y}$.
- **Targeted:** $L(x) = -e(y, \hat{y}) + e(y, y_{\text{target}})$ — also **pull** the output toward a chosen class.

Here $e(\cdot,\cdot)$ is the cross-entropy and $d(x_0, x) \le \varepsilon$ is the **imperceptibility constraint**. FGSM and iterative FGSM are just different optimizers of this one objective.

## 3 · Measuring imperceptibility — L2 vs. L∞

Let $\Delta x = x - x_0$. Two ways to measure its size:

$$\lVert \Delta x \rVert_2 = \sqrt{\sum_i \Delta x_i^2} \qquad \lVert \Delta x \rVert_\infty = \max_i |\Delta x_i|$$

**Four-pixel intuition:** change one pixel a lot, vs. spread the *same* L2 budget thinly across every pixel. Both have equal L2, but the concentrated change is **visible** while the spread-out change is **invisible** — exactly what L∞ captures, since it grows only when *some single pixel* moves a lot.

Because human perception tolerates many tiny changes but not one large one, adversarial attacks constrain the **L∞** ball: $\lVert \Delta x \rVert_\infty \le \varepsilon$.

In [ ]:
# ---- Equal L2, very different L-infinity: why L-inf matches perceptibility ----
x0_pix = denormalize(x0).clamp(0, 1)  # work in [0,1] pixel space
shape = x0_pix.shape
rng = np.random.default_rng(SEED)

def make_concentrated_perturbation(shape, k, amplitude, rng):
    '''Put a large +/- value on k randomly chosen pixels (large L-inf, tiny support).'''
    n = int(np.prod(shape))
    flat = torch.zeros(n, device=device)
    locs = rng.choice(n, size=k, replace=False)
    signs = rng.choice(np.array([-1.0, 1.0]), size=k)
    for loc, s in zip(locs, signs):
        flat[int(loc)] = amplitude * float(s)
    return flat.view(shape)

def make_spread_perturbation(shape, target_l2, rng):
    '''Spread a tiny +/- value across ALL pixels with a given total L2 (small L-inf).'''
    n = int(np.prod(shape))
    signs = rng.choice(np.array([-1.0, 1.0]), size=n).astype('float32')
    amp = target_l2 / np.sqrt(n)  # so that ||spread||_2 == target_l2
    vec = torch.tensor(signs * amp, device=device, dtype=x0_pix.dtype)
    return vec.view(shape)

K = 4
AMPLITUDE = 0.9
conc = make_concentrated_perturbation(shape, K, AMPLITUDE, rng)
target_l2 = conc.norm(p=2).item()
spread = make_spread_perturbation(shape, target_l2, rng)

x_conc = (x0_pix + conc).clamp(0, 1)
x_spread = (x0_pix + spread).clamp(0, 1)

d_conc = x_conc - x0_pix
d_spread = x_spread - x0_pix
L2_conc, Linf_conc = d_conc.norm(p=2).item(), d_conc.abs().max().item()
L2_spread, Linf_spread = d_spread.norm(p=2).item(), d_spread.abs().max().item()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(to_displayable(normalize(x0_pix))); axes[0].set_title('original'); axes[0].axis('off')
axes[1].imshow(to_displayable(normalize(x_conc)))
axes[1].set_title(f'concentrated ({K} px)\nL2={L2_conc:.2f}  Linf={Linf_conc:.2f}'); axes[1].axis('off')
axes[2].imshow(to_displayable(normalize(x_spread)))
axes[2].set_title(f'spread (all px)\nL2={L2_spread:.2f}  Linf={Linf_spread:.4f}'); axes[2].axis('off')
plt.suptitle('Equal L2, very different L-infinity -> only the concentrated change is visible')
plt.tight_layout()
l2_vs_linf_figure = fig
plt.show()

if abs(L2_conc - L2_spread) / max(L2_conc, 1e-8) > 0.1:
    print('[note] clamping shifted the L2 parity slightly; the values shown are the realized norms.')
print(f'L2: conc={L2_conc:.3f}, spread={L2_spread:.3f}  |  Linf: conc={Linf_conc:.3f}, spread={Linf_spread:.5f}')

## 4 · The Fast Gradient Sign Method (FGSM)

FGSM is the canonical **one-step, white-box** attack. Take the **sign** of the gradient of the loss w.r.t. the input and step by ε:

$$x = x_0 - \varepsilon \cdot \operatorname{sign}(\nabla_x L)$$

(In code we equivalently *ascend* the cross-entropy to the true label.) Using only the **sign** lands exactly on a **corner of the L∞ ε-box**, spending the full budget on every pixel. One backprop to the input — simple, fast, and a direct optimizer of the non-targeted objective from Part 2.

In [ ]:
# ---- FGSM: one-step white-box attack in [0,1] pixel space ----
def fgsm(model, x0, y_hat, eps):
    '''
    Single-step Fast Gradient Sign Method.
    x0: NORMALIZED input (1,3,224,224); y_hat: reference class index; eps: L-inf budget in pixel units.
    Non-targeted: ASCEND the cross-entropy to y_hat (= descend the lecture L = -e(y, y_hat)),
    so we step along +sign(grad). Returns a NORMALIZED adversarial tensor.
    '''
    if eps <= 0:
        return x0.detach()  # no-op so the C14 slider at eps=0 shows the benign image

    x_pix = denormalize(x0).clamp(0, 1).detach().clone()
    x_pix.requires_grad_(True)

    logits = model(normalize(x_pix))
    loss = F.cross_entropy(logits, torch.tensor([y_hat], device=device))

    model.zero_grad(set_to_none=True)
    loss.backward()
    assert x_pix.grad is not None, 'no gradient on the input - check requires_grad / frozen model'

    with torch.no_grad():
        x_adv_pix = (x_pix + eps * x_pix.grad.sign()).clamp(0, 1)
    return normalize(x_adv_pix).detach()

# quick self-test
_t0 = fgsm(model, x0, y_hat, 0.0)
assert torch.allclose(_t0, x0, atol=1e-6)
_t1 = fgsm(model, x0, y_hat, 0.02)
assert (denormalize(_t1) - denormalize(x0)).abs().max().item() <= 0.02 + 1e-5
print('fgsm() ready (eps=0 is a no-op; L-inf budget respected).')

In [ ]:
# ---- Run FGSM at a default epsilon and reveal the attack ----
EPS_DEFAULT = 0.02   # L-inf budget in [0,1] pixel units
AMP = 50             # perturbation display amplification

x_adv_fgsm = fgsm(model, x0, y_hat, EPS_DEFAULT)

b_labels, b_probs, _ = predict(x0)
a_labels, a_probs, _ = predict(x_adv_fgsm)

delta = denormalize(x_adv_fgsm) - denormalize(x0)
delta_vis = (delta * AMP + 0.5).clamp(0, 1)  # +0.5 centers the signed perturbation as mid-gray

fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
axes[0].imshow(to_displayable(x0))
axes[0].set_title(f'original\n{b_labels[0]} ({b_probs[0]:.2f})'); axes[0].axis('off')
axes[1].imshow(delta_vis.squeeze(0).permute(1, 2, 0).cpu().numpy())
axes[1].set_title(f'perturbation x{AMP}'); axes[1].axis('off')
axes[2].imshow(to_displayable(x_adv_fgsm))
axes[2].set_title(f'adversarial\n{a_labels[0]} ({a_probs[0]:.2f})'); axes[2].axis('off')
plt.suptitle(f'FGSM at eps={EPS_DEFAULT} (L-inf={delta.abs().max().item():.3f})')
plt.tight_layout()
fgsm_attack_figure = fig
plt.show()

flipped = a_labels[0] != b_labels[0]
print(f'benign: {b_labels[0]} ({b_probs[0]:.2f})  ->  adversarial: {a_labels[0]} ({a_probs[0]:.2f})')
if flipped:
    print(f'Label flipped with an L-inf perturbation of only {delta.abs().max().item():.3f}.')
else:
    print('[note] Label did not flip at the default eps - raise epsilon with the slider in the next cell.')

In [ ]:
# ---- Interactive: epsilon explorer (with a guaranteed static default) ----
def render_fgsm(eps):
    x_adv = fgsm(model, x0, y_hat, eps)
    labels_, probs_, _ = predict(x_adv)
    delta = denormalize(x_adv) - denormalize(x0)
    linf = delta.abs().max().item()
    delta_vis = (delta * AMP + 0.5).clamp(0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
    axes[0].imshow(to_displayable(x0))
    axes[0].set_title(f'original\n{imagenet_labels[y_hat]}'); axes[0].axis('off')
    axes[1].imshow(delta_vis.squeeze(0).permute(1, 2, 0).cpu().numpy())
    axes[1].set_title(f'perturbation x{AMP}\nL-inf={linf:.3f}'); axes[1].axis('off')
    axes[2].imshow(to_displayable(x_adv))
    axes[2].set_title(f'adversarial\n{labels_[0]} ({probs_[0]:.2f})'); axes[2].axis('off')
    plt.suptitle(f'FGSM  |  eps={eps:.3f}')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

fgsm_widget = None
if HAS_WIDGETS:
    _eps_slider = widgets.FloatSlider(value=EPS_DEFAULT, min=0.0, max=0.10, step=0.005,
                                      description='epsilon', continuous_update=False, readout_format='.3f')
    fgsm_widget = widgets.interact(render_fgsm, eps=_eps_slider)
else:
    render_fgsm(EPS_DEFAULT)

## 5 · Iterative FGSM (Basic Iterative Method)

Instead of one big jump, take **many small signed steps** and, after each, **project** back into the L∞ ε-ball around $x_0$ (the lecture's `fix()`):

$$x^{t} = \operatorname{clip}_\varepsilon\big( x^{t-1} - \alpha \cdot \operatorname{sign}(\nabla_x L) \big)$$

Each step moves by a small $\alpha$; the projection clamps every pixel back into $[x_0 - \varepsilon,\, x_0 + \varepsilon]$. Following the loss surface in small clipped steps usually beats one big jump, so at the **same ε** iterative FGSM is a **stronger** attack.

In [ ]:
# ---- Iterative FGSM (BIM) with explicit L-infinity projection + optional targeted mode ----
def ifgsm(model, x0, y_hat, eps, alpha, steps, target=None):
    '''
    Basic Iterative Method: many small signed steps, each projected back into the
    L-inf eps-ball around x0. Returns (normalized adv tensor, trajectory dict).
    target=None -> non-targeted (ascend loss to y_hat); target=idx -> targeted (descend loss to target).
    '''
    steps = max(1, int(steps))
    if alpha <= 0:
        alpha = eps / steps
    if alpha > eps:
        alpha = eps  # keep each step inside the ball

    x0_pix = denormalize(x0).clamp(0, 1).detach()
    x_pix = x0_pix.clone()
    traj = {'step': [], 'loss': [], 'true_conf': [], 'target_conf': []}

    for t in range(steps):
        x_pix.requires_grad_(True)
        logits = model(normalize(x_pix))
        if target is None:
            loss = F.cross_entropy(logits, torch.tensor([y_hat], device=device))
            direction = 1.0   # ascend true-label loss (push away from y_hat)
        else:
            loss = F.cross_entropy(logits, torch.tensor([target], device=device))
            direction = -1.0  # descend target loss (pull toward target)

        model.zero_grad(set_to_none=True)
        loss.backward()
        grad = x_pix.grad

        with torch.no_grad():
            x_pix = x_pix + alpha * direction * grad.sign()
            x_pix = torch.max(torch.min(x_pix, x0_pix + eps), x0_pix - eps).clamp(0, 1)
            probs = F.softmax(model(normalize(x_pix)), dim=1)[0]
            traj['step'].append(t)
            traj['loss'].append(float(loss.item()))
            traj['true_conf'].append(float(probs[y_hat].item()))
            traj['target_conf'].append(float(probs[target].item()) if target is not None else None)
        x_pix = x_pix.detach()

    return normalize(x_pix).detach(), traj

# quick self-test (non-targeted)
_adv, _traj = ifgsm(model, x0, y_hat, eps=0.02, alpha=0.005, steps=5)
assert (denormalize(_adv) - denormalize(x0)).abs().max().item() <= 0.02 + 1e-5
assert len(_traj['step']) == 5
print('ifgsm() ready (projection holds; trajectory recorded).')

In [ ]:
# ---- Interactive: FGSM vs iterative at matched epsilon (static default included) ----
def render_compare(eps, alpha, steps):
    x_fg = fgsm(model, x0, y_hat, eps)
    x_it, traj = ifgsm(model, x0, y_hat, eps, alpha, steps)
    fg_labels, fg_probs, _ = predict(x_fg)
    it_labels, it_probs, _ = predict(x_it)
    linf_fg = (denormalize(x_fg) - denormalize(x0)).abs().max().item()
    linf_it = (denormalize(x_it) - denormalize(x0)).abs().max().item()

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    axes[0].imshow(to_displayable(x_fg))
    axes[0].set_title(f'FGSM (1 step)\n{fg_labels[0]} ({fg_probs[0]:.2f})\nL-inf={linf_fg:.3f}'); axes[0].axis('off')
    axes[1].imshow(to_displayable(x_it))
    axes[1].set_title(f'iterative ({steps} steps)\n{it_labels[0]} ({it_probs[0]:.2f})\nL-inf={linf_it:.3f}'); axes[1].axis('off')
    axes[2].plot(traj['step'], traj['true_conf'], marker='o', color='#C44E52')
    axes[2].set_title('iterative: true-class confidence')
    axes[2].set_xlabel('step'); axes[2].set_ylabel(f'P({imagenet_labels[y_hat]})')
    axes[2].set_ylim(0, 1)
    plt.suptitle(f'Same eps={eps:.3f}: iterative usually drops the true class faster than FGSM')
    plt.tight_layout()
    plt.show()
    globals()['convergence_curve_figure'] = fig
    plt.close(fig)

convergence_curve_figure = None
comparison_widget = None
if HAS_WIDGETS:
    _eps_s = widgets.FloatSlider(value=EPS_DEFAULT, min=0.0, max=0.10, step=0.005, description='eps', continuous_update=False, readout_format='.3f')
    _alpha_s = widgets.FloatSlider(value=0.005, min=0.001, max=0.02, step=0.001, description='alpha', continuous_update=False, readout_format='.3f')
    _steps_s = widgets.IntSlider(value=10, min=1, max=40, step=1, description='steps', continuous_update=False)
    comparison_widget = widgets.interact(render_compare, eps=_eps_s, alpha=_alpha_s, steps=_steps_s)
else:
    render_compare(EPS_DEFAULT, 0.005, 10)

## Targeted attacks

Recall the **targeted** objective from Part 2 — also pull the output **toward** a chosen class $y_{\text{target}}$. We adapt iterative FGSM to **descend the targeted loss**, stepping along $-\operatorname{sign}(\nabla_x L_{\text{target}})$ to force a specific label and reproduce the lecture's *Tiger Cat → Keyboard* (p ≈ 1.0) effect. Pick a target below and watch the benign image get pushed into an arbitrary class.

In [ ]:
# ---- Interactive: targeted attack (dropdown + run button) with a static default ----
EPS_TARGETED = 0.03
ALPHA_TARGETED = 0.005
STEPS_TARGETED = 30

def resolve_label_index(name, labels):
    '''Case-insensitive substring lookup of a class name -> index, or None if absent.'''
    name = name.lower()
    for i, lab in enumerate(labels):
        if name in lab.lower():
            return i
    return None

# Curated candidate targets, resolved by NAME (never hard-coded indices).
_candidates = ['starfish', 'keyboard', 'banana', 'soccer ball', 'lemon', 'broccoli']
target_options = []
_missing = []
for _nm in _candidates:
    _i = resolve_label_index(_nm, imagenet_labels)
    if _i is None:
        _missing.append(_nm)
    else:
        target_options.append((f'{imagenet_labels[_i]} ({_i})', _i))
if _missing:
    print(f'[note] could not resolve, dropped: {_missing}')
if not target_options:
    target_options = [(f'{imagenet_labels[0]} (0)', 0)]  # guaranteed-present fallback

x_adv_targeted = None

def run_targeted(target_idx):
    global x_adv_targeted
    x_adv, traj = ifgsm(model, x0, y_hat, eps=EPS_TARGETED, alpha=ALPHA_TARGETED, steps=STEPS_TARGETED, target=target_idx)
    x_adv_targeted = x_adv
    labels_, probs_, _ = predict(x_adv)
    tconf = traj['target_conf'][-1]

    fig, (ax_img, ax_curve) = plt.subplots(1, 2, figsize=(11, 4.2))
    ax_img.imshow(to_displayable(x_adv))
    ax_img.set_title(f'forced -> {imagenet_labels[target_idx]}\ntop-1: {labels_[0]} ({probs_[0]:.2f})'); ax_img.axis('off')
    ax_curve.plot(traj['step'], traj['target_conf'], marker='o', color='#55A868')
    ax_curve.set_title('target-class confidence')
    ax_curve.set_xlabel('step'); ax_curve.set_ylabel(f'P({imagenet_labels[target_idx]})')
    ax_curve.set_ylim(0, 1)
    plt.suptitle(f'Targeted iterative FGSM: {imagenet_labels[y_hat]} -> {imagenet_labels[target_idx]}')
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    print(f'target {imagenet_labels[target_idx]}: final confidence {tconf:.3f}')

targeted_attack_widget = None
_default_target = next((idx for nm, idx in target_options if 'starfish' in nm.lower()), target_options[0][1])
if HAS_WIDGETS:
    _dd = widgets.Dropdown(options=target_options, value=_default_target, description='target')
    targeted_attack_widget = widgets.interact_manual(run_targeted, target_idx=_dd)

# Always produce a static result so x_adv_targeted is defined under Run All.
run_targeted(_default_target)

## 6 · Passive defense via input transformation

A **passive** defense leaves the model untouched and inserts a **training-free transform before the classifier** (Filter → Network). Smoothing, JPEG-style compression, or random resize/pad **destroys the fragile high-frequency attack signal** while the benign content survives, so the attacked *Keyboard* image is classified as *Tiger Cat* again.

**The catch — a robustness/accuracy trade-off:** the same filter also blurs clean images, lowering confidence even with no attack (the lecture's *0.64 → 0.45*). More filtering buys robustness at the cost of clean accuracy.

In [ ]:
# ---- Passive defense: a training-free input transform (Gaussian smoothing) ----
def defend(x, strength):
    '''Smooth a NORMALIZED image to destroy fragile high-frequency attack signal. Returns normalized.'''
    xp = denormalize(x).clamp(0, 1)
    if strength <= 0:
        return normalize(xp).detach()  # near-identity
    ksize = max(3, 2 * int(strength) + 1)  # gaussian_blur needs an odd kernel >= 3
    sigma = max(float(strength), 0.1)
    try:
        filtered = TF.gaussian_blur(xp, kernel_size=[ksize, ksize], sigma=[sigma, sigma])
    except Exception:
        if HAS_SCIPY:
            arr = xp.squeeze(0).cpu().numpy()
            for c in range(arr.shape[0]):
                arr[c] = ndi.gaussian_filter(arr[c], sigma=sigma)
            filtered = torch.tensor(arr, device=device).unsqueeze(0)
        else:
            filtered = F.avg_pool2d(xp, kernel_size=3, stride=1, padding=1)
    filtered = filtered.clamp(0, 1)
    return normalize(filtered).detach()

DEFENSE_STRENGTH_DEFAULT = 2.0

before_l, before_p, _ = predict(x_adv_fgsm)
x_def = defend(x_adv_fgsm, DEFENSE_STRENGTH_DEFAULT)
after_l, after_p, _ = predict(x_def)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(to_displayable(x_adv_fgsm))
axes[0].set_title(f'attacked\n{before_l[0]} ({before_p[0]:.2f})'); axes[0].axis('off')
axes[1].imshow(to_displayable(x_def))
axes[1].set_title(f'defended (smoothed)\n{after_l[0]} ({after_p[0]:.2f})'); axes[1].axis('off')
plt.suptitle(f'Passive defense recovers toward {imagenet_labels[y_hat]}')
plt.tight_layout()
defense_recovery_figure = fig
plt.show()

print(f'attacked: {before_l[0]} ({before_p[0]:.2f})  ->  defended: {after_l[0]} ({after_p[0]:.2f})')
if after_l[0] != imagenet_labels[y_hat]:
    print('[note] default strength did not fully recover the label - increase strength with the slider in the next cell.')

In [ ]:
# ---- Interactive: robustness/accuracy trade-off of the passive filter ----
def render_tradeoff(strength):
    clean_l, clean_p, _ = predict(x0)
    cleanf = defend(x0, strength)
    cleanf_l, cleanf_p, _ = predict(cleanf)
    att_l, att_p, _ = predict(x_adv_fgsm)
    attf = defend(x_adv_fgsm, strength)
    attf_l, attf_p, _ = predict(attf)

    fig, axes = plt.subplots(2, 2, figsize=(9, 8))
    axes[0, 0].imshow(to_displayable(x0)); axes[0, 0].set_title(f'clean\n{clean_l[0]} ({clean_p[0]:.2f})'); axes[0, 0].axis('off')
    axes[0, 1].imshow(to_displayable(cleanf)); axes[0, 1].set_title(f'clean + filter\n{cleanf_l[0]} ({cleanf_p[0]:.2f})'); axes[0, 1].axis('off')
    axes[1, 0].imshow(to_displayable(x_adv_fgsm)); axes[1, 0].set_title(f'attacked\n{att_l[0]} ({att_p[0]:.2f})'); axes[1, 0].axis('off')
    axes[1, 1].imshow(to_displayable(attf)); axes[1, 1].set_title(f'attacked + filter\n{attf_l[0]} ({attf_p[0]:.2f})'); axes[1, 1].axis('off')
    plt.suptitle(f'filter strength={strength:.1f}  |  clean conf {clean_p[0]:.2f} -> {cleanf_p[0]:.2f}  (robustness/accuracy trade-off)')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

defense_tradeoff_widget = None
if HAS_WIDGETS:
    _s_slider = widgets.FloatSlider(value=DEFENSE_STRENGTH_DEFAULT, min=0.0, max=5.0, step=0.5, description='filter strength', continuous_update=False)
    defense_tradeoff_widget = widgets.interact(render_tradeoff, strength=_s_slider)
else:
    render_tradeoff(DEFENSE_STRENGTH_DEFAULT)

## Recap & where to go next

One continuous storyline — **define → formalize → measure → attack → strengthen → defend**:

- **LO1** — an adversarial example is a tiny perturbation that flips a **fixed** classifier; attacks are *targeted* or *non-targeted*.
- **LO2** — the attack is **constrained optimization** of an input-space loss, $\arg\min_{d(x_0,x)\le\varepsilon} L(x)$.
- **LO3** — **L∞** matches imperceptibility better than L2 (equal L2 ≠ equal visibility).
- **LO4** — **FGSM** is a one-step, white-box, sign-of-gradient attack.
- **LO5** — **iterative FGSM** with ε-ball projection is stronger at the same ε.
- **LO6** — a **passive filter** defends but **trades off clean accuracy**.

**Limits of passive defense:** a filter is easy to bypass once the attacker knows it. Intentionally left out to keep this notebook fast and executable:

- **Black-box transferability** — attacks crafted on a surrogate/ensemble transfer to unseen models ([Papernot et al., 2016](https://arxiv.org/abs/1605.07277)).
- **Proactive defense / adversarial training** ([Madry et al., 2017](https://arxiv.org/abs/1706.06083)).
- **The geometric “why”** — *Adversarial Examples Are Not Bugs, They Are Features* ([Ilyas et al., 2019](https://arxiv.org/abs/1905.02175)).
- **Physical & multimodal attacks**, e.g. adversarial patches ([Brown et al., 2017](https://arxiv.org/abs/1712.09665)).